#### Setup

In [1]:
import boto3
import json
from pathlib import Path
import sys

sys.path.append("../src")

from chunking import split_text_into_chunks

#### S3 paths

In [2]:
BUCKET_NAME = "YOUR_BUCKET_NAME"

CLEAN_TEXT_PREFIX = "papers/clean_text/"
CHUNKS_PREFIX = "chunks/"

s3 = boto3.client("s3")

#### List cleaned text files

In [3]:
response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=CLEAN_TEXT_PREFIX
)

clean_files = [
    obj["Key"]
    for obj in response.get("Contents", [])
    if obj["Key"].lower().endswith(".txt")
]

print(f"Found {len(clean_files)} cleaned text files.")

Found 34 cleaned text files.


#### Chunk all papers and save JSONL to S3

In [4]:
all_chunks = []

for key in clean_files:
    filename = Path(key).name

    obj = s3.get_object(
        Bucket=BUCKET_NAME,
        Key=key
    )

    text = obj["Body"].read().decode("utf-8", errors="ignore")

    chunks = split_text_into_chunks(
        text=text,
        source_file=filename,
        chunk_size_words=900,
        overlap_words=150,
    )

    all_chunks.extend(chunks)

    print(f"{filename}: {len(chunks)} chunks")

BindVAE.txt: 17 chunks
CASTLE.txt: 16 chunks
CAVACHON.txt: 11 chunks
Chromatin_GeneRegulation_Review.txt: 17 chunks
Cicero.txt: 31 chunks
GNODEVAE.txt: 27 chunks
GenKI.txt: 18 chunks
JAMIE.txt: 15 chunks
LiVAE.txt: 19 chunks
MultiVI.txt: 16 chunks
Papers___MDPI_GENES___GAGAM_v1_2__an_improvement_on_peak_labeling_and_Genomic_Annotated_Gene_Activity_Matrix_construction.txt: 17 chunks
SCA.txt: 9 chunks
UnionCom.txt: 12 chunks
VAE_BatchCorrection_scRNAseq_Benchmark.txt: 49 chunks
biVI.txt: 10 chunks
cobolt.txt: 10 chunks
factVAE.txt: 10 chunks
hybridVI.txt: 29 chunks
pair.txt: 20 chunks
peakVI.txt: 16 chunks
phd-ConvNet-VAE.txt: 57 chunks
phd-scPair.txt: 74 chunks
review1.txt: 19 chunks
review2.txt: 8 chunks
salirex.txt: 10 chunks
scAMACE.txt: 10 chunks
scButterfly.txt: 23 chunks
scJVAE.txt: 14 chunks
scMVAE.txt: 13 chunks
scVAE.txt: 12 chunks
scVAG.txt: 12 chunks
scVAGATE.txt: 12 chunks
scVIDR.txt: 14 chunks
sciCAN.txt: 13 chunks


#### Save all chunks

In [5]:
jsonl_data = "\n".join(
    json.dumps(chunk, ensure_ascii=False)
    for chunk in all_chunks
)

output_key = CHUNKS_PREFIX + "paper_chunks.jsonl"

s3.put_object(
    Bucket=BUCKET_NAME,
    Key=output_key,
    Body=jsonl_data.encode("utf-8"),
    ContentType="application/jsonl"
)

print(f"Saved {len(all_chunks)} chunks to s3://{BUCKET_NAME}/{output_key}")

Saved 660 chunks to s3://multiomic-vae-literature-rag-123223178042-eu-north-1-an/chunks/paper_chunks.jsonl


#### Verify

In [6]:
response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=CHUNKS_PREFIX
)

for obj in response.get("Contents", []):
    print(obj["Key"], obj["Size"])

chunks/ 0
chunks/paper_chunks.jsonl 3973807
